# NeuroScan - Dyslexia Detection Model Training
## Using Handwritten Letter Analysis (Reversal Detection)

**Dataset:** Dyslexia Handwriting Dataset with 3 classes:
- **Reversal** - Letters written reversed (b/d, p/q confusion) → HIGH risk
- **Corrected** - Previously reversed, now corrected → MODERATE risk
- **Normal** - Normal letter writing → LOW risk

**Instructions:**
1. Runtime → Change runtime type → **T4 GPU**
2. Run cells in order
3. Download trained model at the end

In [ ]:
# Cell 1: Install dependencies
!pip install tensorflow opencv-python-headless scikit-learn seaborn -q
print("✅ Dependencies installed!")

In [ ]:
# Cell 2: Check GPU & Import libraries
import tensorflow as tf
import numpy as np
import cv2
import os
import json
from pathlib import Path
import matplotlib.pyplot as plt
from collections import Counter

print(f"TensorFlow: {tf.__version__}")
gpus = tf.config.list_physical_devices('GPU')
if gpus:
    for gpu in gpus:
        tf.config.experimental.set_memory_growth(gpu, True)
    print(f"✅ GPU: {gpus[0]}")
else:
    print("⚠️ No GPU! Runtime → Change runtime type → T4 GPU")

In [ ]:
# Cell 3: Setup Kaggle & Download Dataset
import os

# Your Kaggle credentials
os.environ['KAGGLE_USERNAME'] = 'YOUR_USERNAME'  # <-- Enter your Kaggle username
os.environ['KAGGLE_KEY'] = 'KGAT_4b2e044641e8331cfaf834407694e690'

!pip install kaggle -q

print("📥 Downloading dataset...")
!kaggle datasets download -d drizasazanitaisa/dyslexia-handwriting-dataset -p data --unzip -q
print("✅ Downloaded!")

In [ ]:
# Cell 4: Extract RAR file (password protected)
!apt-get install unrar -qq

import glob

# Find RAR file
rar_files = glob.glob('data/**/*.rar', recursive=True)
if rar_files:
    rar_file = rar_files[0]
    password = 'WanAsy321'
    print(f"📦 Extracting: {rar_file}")
    !unrar x -p{password} "{rar_file}" data/ -o+
    os.remove(rar_file)
    print("✅ Extracted!")
else:
    print("No RAR file found - data may already be extracted")

# Show structure
print("\n=== Dataset Structure ===")
!find data -type d | head -15

In [ ]:
# Cell 5: Configuration

# Dataset paths - adjust if your folder name is different
BASE_PATH = 'data/Gambo'  # or 'data/gambo' - check Cell 4 output

# Check if path exists, try alternatives
if not os.path.exists(BASE_PATH):
    alternatives = ['data/Gambo', 'data/gambo', 'data/GAMBO', 
                    'data/Dataset', 'data/dataset']
    for alt in alternatives:
        if os.path.exists(alt):
            BASE_PATH = alt
            break
    else:
        # Find any folder with train/test subdirs
        for root, dirs, files in os.walk('data'):
            if 'train' in dirs or 'Train' in dirs:
                BASE_PATH = root
                break

print(f"📁 Using base path: {BASE_PATH}")

# Find train and test folders
TRAIN_PATH = None
TEST_PATH = None

for item in os.listdir(BASE_PATH):
    item_lower = item.lower()
    full_path = os.path.join(BASE_PATH, item)
    if os.path.isdir(full_path):
        if 'train' in item_lower:
            TRAIN_PATH = full_path
        elif 'test' in item_lower:
            TEST_PATH = full_path

print(f"📁 Train path: {TRAIN_PATH}")
print(f"📁 Test path: {TEST_PATH}")

# Model config
IMG_SIZE = 64  # Letters are small, 64x64 is enough
BATCH_SIZE = 32
EPOCHS = 50
LEARNING_RATE = 0.001

# Class mapping for 3-class classification
# Reversal=2 (HIGH risk), Corrected=1 (MODERATE), Normal=0 (LOW)
CLASS_MAPPING = {
    'reversal': 2,    # HIGH dyslexia risk
    'corrected': 1,   # MODERATE risk (corrected but had issues)
    'normal': 0       # LOW risk (normal writing)
}

CLASS_NAMES = ['Normal', 'Corrected', 'Reversal']
NUM_CLASSES = 3

print(f"\n✅ Config ready!")
print(f"   Image size: {IMG_SIZE}x{IMG_SIZE}")
print(f"   Classes: {CLASS_NAMES}")

In [ ]:
# Cell 6: Load and preprocess images

def load_image(path, size=IMG_SIZE):
    """Load and preprocess a single B&W letter image."""
    try:
        # Read image
        img = cv2.imread(str(path), cv2.IMREAD_GRAYSCALE)
        if img is None:
            return None
        
        # Resize
        img = cv2.resize(img, (size, size))
        
        # Normalize to [0, 1]
        img = img.astype(np.float32) / 255.0
        
        # Add channel dimension (grayscale)
        img = np.expand_dims(img, axis=-1)
        
        return img
    except Exception as e:
        return None

def load_dataset(base_path):
    """Load all images from train or test folder."""
    images = []
    labels = []
    
    if base_path is None:
        print("❌ Path is None!")
        return np.array([]), np.array([])
    
    # Iterate through class folders
    for folder_name in os.listdir(base_path):
        folder_path = os.path.join(base_path, folder_name)
        
        if not os.path.isdir(folder_path):
            continue
        
        # Get class label
        folder_lower = folder_name.lower()
        label = None
        for class_name, class_id in CLASS_MAPPING.items():
            if class_name in folder_lower:
                label = class_id
                break
        
        if label is None:
            print(f"  ⚠️ Unknown folder: {folder_name}, skipping")
            continue
        
        # Load images from this folder
        count = 0
        for img_file in os.listdir(folder_path):
            if img_file.lower().endswith(('.png', '.jpg', '.jpeg', '.bmp')):
                img_path = os.path.join(folder_path, img_file)
                img = load_image(img_path)
                if img is not None:
                    images.append(img)
                    labels.append(label)
                    count += 1
        
        print(f"  📁 {folder_name}: {count} images (class {label}: {CLASS_NAMES[label]})")
    
    return np.array(images), np.array(labels)

# Load training data
print("=== Loading Training Data ===")
X_train, y_train = load_dataset(TRAIN_PATH)
print(f"\n✅ Training set: {len(X_train)} images")

# Load test data
print("\n=== Loading Test Data ===")
X_test, y_test = load_dataset(TEST_PATH)
print(f"\n✅ Test set: {len(X_test)} images")

# Class distribution
print("\n=== Class Distribution ===")
print("Training:")
for i, name in enumerate(CLASS_NAMES):
    count = np.sum(y_train == i)
    print(f"  {name}: {count} ({count/len(y_train)*100:.1f}%)")

print("\nTest:")
for i, name in enumerate(CLASS_NAMES):
    count = np.sum(y_test == i)
    print(f"  {name}: {count} ({count/len(y_test)*100:.1f}%)")

In [ ]:
# Cell 7: Visualize sample images

fig, axes = plt.subplots(3, 6, figsize=(15, 8))
fig.suptitle('Sample Images by Class', fontsize=14)

for row, (class_id, class_name) in enumerate(enumerate(CLASS_NAMES)):
    indices = np.where(y_train == class_id)[0][:6]
    for col, idx in enumerate(indices):
        axes[row, col].imshow(X_train[idx].squeeze(), cmap='gray')
        axes[row, col].set_title(f'{class_name}', 
                                  color='red' if class_id==2 else ('orange' if class_id==1 else 'green'))
        axes[row, col].axis('off')

plt.tight_layout()
plt.show()

print("🔴 Reversal = HIGH risk (reversed letters like b→d)")
print("🟠 Corrected = MODERATE risk (had issues, now corrected)")
print("🟢 Normal = LOW risk (normal writing)")

In [ ]:
# Cell 8: Create validation split from training data
from sklearn.model_selection import train_test_split

X_train_final, X_val, y_train_final, y_val = train_test_split(
    X_train, y_train,
    test_size=0.15,
    random_state=42,
    stratify=y_train
)

print(f"Training:   {len(X_train_final)} samples")
print(f"Validation: {len(X_val)} samples")
print(f"Test:       {len(X_test)} samples")

In [ ]:
# Cell 9: Build CNN Model for Letter Classification
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import (
    Conv2D, MaxPooling2D, Dense, Dropout, Flatten,
    BatchNormalization, GlobalAveragePooling2D
)
from tensorflow.keras.optimizers import Adam

def build_model(input_shape, num_classes):
    """Build CNN optimized for letter image classification."""
    model = Sequential([
        # Block 1 - Extract basic features
        Conv2D(32, (3, 3), activation='relu', padding='same', input_shape=input_shape),
        BatchNormalization(),
        Conv2D(32, (3, 3), activation='relu', padding='same'),
        BatchNormalization(),
        MaxPooling2D((2, 2)),
        Dropout(0.25),
        
        # Block 2 - Extract letter shapes
        Conv2D(64, (3, 3), activation='relu', padding='same'),
        BatchNormalization(),
        Conv2D(64, (3, 3), activation='relu', padding='same'),
        BatchNormalization(),
        MaxPooling2D((2, 2)),
        Dropout(0.25),
        
        # Block 3 - Detect reversal patterns
        Conv2D(128, (3, 3), activation='relu', padding='same'),
        BatchNormalization(),
        Conv2D(128, (3, 3), activation='relu', padding='same'),
        BatchNormalization(),
        MaxPooling2D((2, 2)),
        Dropout(0.25),
        
        # Global pooling instead of flatten (reduces params)
        GlobalAveragePooling2D(),
        
        # Dense layers for classification
        Dense(256, activation='relu'),
        BatchNormalization(),
        Dropout(0.5),
        Dense(64, activation='relu'),
        Dropout(0.3),
        
        # Output: 3 classes (Normal, Corrected, Reversal)
        Dense(num_classes, activation='softmax')
    ])
    
    return model

# Build model
input_shape = (IMG_SIZE, IMG_SIZE, 1)  # Grayscale
model = build_model(input_shape, NUM_CLASSES)

model.compile(
    optimizer=Adam(learning_rate=LEARNING_RATE),
    loss='sparse_categorical_crossentropy',  # For integer labels
    metrics=['accuracy']
)

print("✅ Model built!")
model.summary()

In [ ]:
# Cell 10: Data Augmentation for handwritten letters
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.callbacks import EarlyStopping, ModelCheckpoint, ReduceLROnPlateau

# Augmentation suitable for handwritten letters
train_datagen = ImageDataGenerator(
    rotation_range=10,        # Slight rotation (handwriting varies)
    width_shift_range=0.1,    # Small shifts
    height_shift_range=0.1,
    zoom_range=0.1,           # Slight zoom
    shear_range=0.1,          # Shear for slant variation
    fill_mode='constant',
    cval=1.0                  # Fill with white (background)
)

# Callbacks
callbacks = [
    EarlyStopping(
        monitor='val_loss',
        patience=10,
        restore_best_weights=True,
        verbose=1
    ),
    ModelCheckpoint(
        'best_dyslexia_model.keras',
        monitor='val_accuracy',
        save_best_only=True,
        verbose=1
    ),
    ReduceLROnPlateau(
        monitor='val_loss',
        factor=0.5,
        patience=5,
        min_lr=1e-7,
        verbose=1
    )
]

print("✅ Data augmentation and callbacks ready!")

In [ ]:
# Cell 11: TRAIN THE MODEL
print("="*60)
print("🚀 TRAINING DYSLEXIA DETECTION MODEL")
print("="*60)
print(f"Training samples: {len(X_train_final)}")
print(f"Validation samples: {len(X_val)}")
print(f"Epochs: {EPOCHS}, Batch size: {BATCH_SIZE}")
print("="*60)

# Create data generator
train_generator = train_datagen.flow(
    X_train_final, y_train_final,
    batch_size=BATCH_SIZE
)

# Train
history = model.fit(
    train_generator,
    steps_per_epoch=len(X_train_final) // BATCH_SIZE,
    epochs=EPOCHS,
    validation_data=(X_val, y_val),
    callbacks=callbacks,
    verbose=1
)

print("\n✅ Training complete!")

In [ ]:
# Cell 12: Plot training history
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Loss
axes[0].plot(history.history['loss'], label='Train', linewidth=2)
axes[0].plot(history.history['val_loss'], label='Validation', linewidth=2)
axes[0].set_title('Model Loss', fontsize=12)
axes[0].set_xlabel('Epoch')
axes[0].set_ylabel('Loss')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

# Accuracy
axes[1].plot(history.history['accuracy'], label='Train', linewidth=2)
axes[1].plot(history.history['val_accuracy'], label='Validation', linewidth=2)
axes[1].set_title('Model Accuracy', fontsize=12)
axes[1].set_xlabel('Epoch')
axes[1].set_ylabel('Accuracy')
axes[1].legend()
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('training_history.png', dpi=150)
plt.show()

print(f"\n📊 Best validation accuracy: {max(history.history['val_accuracy']):.4f}")

In [ ]:
# Cell 13: Evaluate on Test Set
from sklearn.metrics import classification_report, confusion_matrix
import seaborn as sns

print("=== Evaluating on Test Set ===")

# Get predictions
y_pred_proba = model.predict(X_test, verbose=0)
y_pred = np.argmax(y_pred_proba, axis=1)

# Classification report
print("\n📊 Classification Report:")
print(classification_report(y_test, y_pred, target_names=CLASS_NAMES))

# Confusion matrix
cm = confusion_matrix(y_test, y_pred)
plt.figure(figsize=(8, 6))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=CLASS_NAMES, yticklabels=CLASS_NAMES)
plt.title('Confusion Matrix', fontsize=14)
plt.ylabel('Actual')
plt.xlabel('Predicted')
plt.tight_layout()
plt.savefig('confusion_matrix.png', dpi=150)
plt.show()

# Overall accuracy
test_loss, test_acc = model.evaluate(X_test, y_test, verbose=0)
print(f"\n✅ Test Accuracy: {test_acc:.4f} ({test_acc*100:.2f}%)")

In [ ]:
# Cell 14: Visualize predictions

# Show some predictions
fig, axes = plt.subplots(3, 6, figsize=(15, 8))
fig.suptitle('Sample Predictions (Actual vs Predicted)', fontsize=14)

# Get random samples
np.random.seed(42)
indices = np.random.choice(len(X_test), 18, replace=False)

for i, idx in enumerate(indices):
    row, col = i // 6, i % 6
    
    ax = axes[row, col]
    ax.imshow(X_test[idx].squeeze(), cmap='gray')
    
    actual = CLASS_NAMES[y_test[idx]]
    predicted = CLASS_NAMES[y_pred[idx]]
    confidence = y_pred_proba[idx][y_pred[idx]] * 100
    
    is_correct = y_test[idx] == y_pred[idx]
    color = 'green' if is_correct else 'red'
    
    ax.set_title(f'A:{actual}\nP:{predicted} ({confidence:.0f}%)', 
                 fontsize=9, color=color)
    ax.axis('off')

plt.tight_layout()
plt.show()

In [ ]:
# Cell 15: Save models for deployment
import json
import shutil

# Create output directory
os.makedirs('trained_models', exist_ok=True)

# Save model in multiple formats
model.save('trained_models/dyslexia_model.keras')
model.save('trained_models/dyslexia_model.h5')

# Save TFLite version for mobile/edge
converter = tf.lite.TFLiteConverter.from_keras_model(model)
tflite_model = converter.convert()
with open('trained_models/dyslexia_model.tflite', 'wb') as f:
    f.write(tflite_model)

# Save configuration
config = {
    'IMG_SIZE': IMG_SIZE,
    'IMG_HEIGHT': IMG_SIZE,
    'IMG_WIDTH': IMG_SIZE,
    'NUM_CLASSES': NUM_CLASSES,
    'CLASS_NAMES': CLASS_NAMES,
    'CLASS_MAPPING': CLASS_MAPPING,
    'test_accuracy': float(test_acc),
    'model_type': '3-class-letter-classification',
    'description': 'Dyslexia detection via letter reversal analysis',
    'risk_mapping': {
        '0': {'name': 'Normal', 'risk': 'LOW', 'score_range': '0-35'},
        '1': {'name': 'Corrected', 'risk': 'MODERATE', 'score_range': '35-70'},
        '2': {'name': 'Reversal', 'risk': 'HIGH', 'score_range': '70-100'}
    }
}

with open('trained_models/config.json', 'w') as f:
    json.dump(config, f, indent=2)

# Copy training artifacts
shutil.copy('training_history.png', 'trained_models/')
shutil.copy('confusion_matrix.png', 'trained_models/')

print("\n✅ Models saved!")
print("\nFiles created:")
for f in sorted(os.listdir('trained_models')):
    size = os.path.getsize(f'trained_models/{f}') / 1024
    unit = 'KB'
    if size > 1024:
        size = size / 1024
        unit = 'MB'
    print(f"  📄 {f} ({size:.1f} {unit})")

In [ ]:
# Cell 16: Download trained models
!zip -r neuroscan_dyslexia_model.zip trained_models/

from google.colab import files
files.download('neuroscan_dyslexia_model.zip')

print("""
========================================
🎉 TRAINING COMPLETE!
========================================

📥 Download 'neuroscan_dyslexia_model.zip'

📁 Extract to: neurascan-python-ai/models/

Your folder should look like:
neurascan-python-ai/
├── models/
│   ├── dyslexia_model.keras
│   ├── dyslexia_model.h5
│   ├── dyslexia_model.tflite
│   └── config.json
└── ml_models.py  (update with integration code)

See next cell for integration code!
""")

In [ ]:
# Cell 17: Test single image prediction

def predict_dyslexia_risk(image_path_or_array):
    """
    Predict dyslexia risk from a handwriting image.
    Returns risk level and confidence score.
    """
    # Load image if path provided
    if isinstance(image_path_or_array, str):
        img = load_image(image_path_or_array)
    else:
        img = image_path_or_array
    
    if img is None:
        return None
    
    # Add batch dimension if needed
    if len(img.shape) == 3:
        img = np.expand_dims(img, axis=0)
    
    # Predict
    proba = model.predict(img, verbose=0)[0]
    predicted_class = np.argmax(proba)
    
    # Calculate dyslexia score (0-100)
    # Weight: Reversal=100, Corrected=50, Normal=0
    dyslexia_score = (proba[2] * 100 + proba[1] * 50 + proba[0] * 0)
    
    # Determine risk level
    if dyslexia_score >= 70:
        risk_level = 'HIGH'
        recommendation = 'Strong reversal patterns detected. Professional evaluation recommended.'
    elif dyslexia_score >= 40:
        risk_level = 'MODERATE'
        recommendation = 'Some indicators detected. Further assessment suggested.'
    else:
        risk_level = 'LOW'
        recommendation = 'Writing patterns within typical range.'
    
    return {
        'dyslexia_score': round(float(dyslexia_score), 2),
        'risk_level': risk_level,
        'predicted_class': CLASS_NAMES[predicted_class],
        'confidence': f"{proba[predicted_class]*100:.1f}%",
        'class_probabilities': {
            'Normal': f"{proba[0]*100:.1f}%",
            'Corrected': f"{proba[1]*100:.1f}%",
            'Reversal': f"{proba[2]*100:.1f}%"
        },
        'recommendation': recommendation,
        'analysis_type': 'Real ML (Letter Reversal Detection)'
    }

# Test on a few samples
print("=== Sample Predictions ===")
for i in range(3):
    idx = np.random.randint(len(X_test))
    result = predict_dyslexia_risk(X_test[idx])
    actual = CLASS_NAMES[y_test[idx]]
    
    print(f"\nSample {i+1}:")
    print(f"  Actual class: {actual}")
    print(f"  Predicted: {result['predicted_class']} ({result['confidence']})")
    print(f"  Dyslexia Score: {result['dyslexia_score']} ({result['risk_level']})")

In [ ]:
# Cell 18: Integration code for NeuroScan project
print("""
# ============================================================
# ADD THIS TO: neurascan-python-ai/ml_models.py
# ============================================================

import tensorflow as tf
import numpy as np
import cv2
import json
import os

MODEL_DIR = os.path.join(os.path.dirname(__file__), 'models')
_model = None
_config = None

def load_trained_model():
    '''Load the trained CNN model (cached).'''
    global _model, _config
    
    if _model is None:
        model_path = os.path.join(MODEL_DIR, 'dyslexia_model.keras')
        if not os.path.exists(model_path):
            model_path = os.path.join(MODEL_DIR, 'dyslexia_model.h5')
        _model = tf.keras.models.load_model(model_path)
        print(f"Loaded model from {model_path}")
    
    if _config is None:
        config_path = os.path.join(MODEL_DIR, 'config.json')
        with open(config_path) as f:
            _config = json.load(f)
    
    return _model, _config


def analyze_handwriting_ml(image_path: str, text: str = '') -> dict:
    '''
    Analyze handwriting for dyslexia indicators using trained CNN.
    Works with full handwriting images by extracting letter regions.
    '''
    model, config = load_trained_model()
    img_size = config.get('IMG_SIZE', 64)
    
    # Load image
    img = cv2.imread(image_path)
    if img is None:
        raise ValueError(f"Could not read image: {image_path}")
    
    # Convert to grayscale
    gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)
    
    # Threshold to find letter regions
    _, thresh = cv2.threshold(gray, 0, 255, cv2.THRESH_BINARY_INV + cv2.THRESH_OTSU)
    
    # Find contours (letter candidates)
    contours, _ = cv2.findContours(thresh, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
    
    # Filter and extract letter regions
    letter_predictions = []
    min_area = 100  # Minimum contour area
    
    for contour in contours:
        x, y, w, h = cv2.boundingRect(contour)
        area = w * h
        
        if area < min_area:
            continue
        
        # Extract letter region
        letter_img = gray[y:y+h, x:x+w]
        
        # Resize to model input size
        letter_img = cv2.resize(letter_img, (img_size, img_size))
        letter_img = letter_img.astype(np.float32) / 255.0
        letter_img = np.expand_dims(letter_img, axis=(0, -1))
        
        # Predict
        proba = model.predict(letter_img, verbose=0)[0]
        letter_predictions.append(proba)
    
    # Aggregate predictions across all detected letters
    if letter_predictions:
        avg_proba = np.mean(letter_predictions, axis=0)
        # Dyslexia score: weighted by class severity
        dyslexia_score = float(avg_proba[2] * 100 + avg_proba[1] * 50)
    else:
        # Fallback: analyze whole image
        resized = cv2.resize(gray, (img_size, img_size))
        resized = resized.astype(np.float32) / 255.0
        resized = np.expand_dims(resized, axis=(0, -1))
        avg_proba = model.predict(resized, verbose=0)[0]
        dyslexia_score = float(avg_proba[2] * 100 + avg_proba[1] * 50)
    
    # Determine risk level
    if dyslexia_score >= 70:
        confidence = 'HIGH'
        recommendation = 'Strong indicators present. Professional evaluation recommended.'
    elif dyslexia_score >= 40:
        confidence = 'MODERATE'
        recommendation = 'Some indicators detected. Further assessment suggested.'
    else:
        confidence = 'LOW'
        recommendation = 'Indicators within typical range.'
    
    # Also calculate dysgraphia using existing feature extractor
    try:
        features = HandwritingFeatureExtractor.extract_features(image_path, text)
        dysgraphia_clf = DysgraphiaClassifier()
        dysgraphia_score, dysgraphia_details = dysgraphia_clf.calculate_risk_score(features)
    except:
        dysgraphia_score = 0.0
        dysgraphia_details = {'confidence': 'UNKNOWN', 'recommendation': 'Could not analyze'}
    
    return {
        'dyslexia_score': round(dyslexia_score, 2),
        'dyslexia_details': {
            'confidence': confidence,
            'recommendation': recommendation,
            'letters_analyzed': len(letter_predictions),
            'class_probabilities': {
                'Normal': f"{avg_proba[0]*100:.1f}%",
                'Corrected': f"{avg_proba[1]*100:.1f}%",
                'Reversal': f"{avg_proba[2]*100:.1f}%"
            }
        },
        'dysgraphia_score': round(dysgraphia_score, 2),
        'dysgraphia_details': dysgraphia_details,
        'analysis_type': 'Real ML Analysis (Letter Reversal CNN)',
    }
""")